# FID score:
The Frechet Inception Distance (FID) is a metric used to evaluate the quality of generated images by comparing them to real images. It works by passing both real and generated images through a pretrained Inception network to extract high-level feature representations. FID then models these features as multivariate Gaussian distributions and computes the Fréchet distance between them. A lower FID score indicates that the generated images are more similar to real images in terms of both visual quality and diversity.

In [2]:
# Switch path to root of project
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"
current_folder = globals()['_dh'][0]
os.chdir(os.path.dirname(os.path.abspath(current_folder)))
%load_ext autoreload
%autoreload 2

In [3]:
import cv2
import numpy as np
import torch
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from eval.utils import RGBFIDDataset, compute_fid
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF

from torchmetrics.image.fid import FrechetInceptionDistance
from torch.utils.data import Dataset, DataLoader
device = "cuda" if torch.cuda.is_available() else "cpu"

## Data formating for FID:

In [4]:
first_rgb = Image.open("/scratch/tmaillar/rgb/train/point_32_view_0_domain_rgb.png").convert("RGB")
second_rgb = Image.open("/scratch/tmaillar/rgb/train/point_40_view_0_domain_rgb.png").convert("RGB")

crop_settings_0 = np.load("/work/com-304/datasets/clevr_com_304/train/crop_settings/00001.npy")

def format_image_for_FID(path, crop_settings, aug_idx = 1, device="cuda"):
    img = Image.open(path).convert("RGB")
    
    x1, y1, x2, y2, flip = crop_settings[aug_idx]
    top, left, h, w = y1, x1, (y2 - y1), (x2 - x1)
    
    cropped = TF.crop(img, top, left, h, w)
    resized = cropped.resize((256, 256), resample=Image.BILINEAR)
    
    if flip:
        resized = TF.hflip(resized)
        
    img = np.array(resized)
    img = torch.from_numpy(img).permute(2,0,1).byte()
    
    resized.show()

    return img.to(device).unsqueeze(0)

## Test with same Data:
**FID score must be ~0**

In [30]:
crop_settings_01_pth = "/work/com-304/datasets/clevr_com_304/train/crop_settings/00001.npy"
real_data_pth = "/scratch/tmaillar/rgb/train"
fake_data_pth = real_data_pth

score = compute_fid(
    real_dir=real_data_pth,
    fake_dir=fake_data_pth,
    crop_settings_dir=crop_settings_01_pth,
    batch_size=32,
    device=device,
    image_size=256,
    num_workers=4,
    max_images=10000
)

print(f"FID score = {score}")

Fake images:  20%|█████████████▍                                                     | 313/1563 [00:20<01:20, 15.46it/s]


FID score = -3.637978807091713e-12
